# Used words for this model prediction

In [ ]:
# model	  ------   Avtomobilning aniq nomi yoki modeli.

# model_index	----------   Modelning indeks yoki kod raqami (ma’lumotlar bazasida identifikatsiya uchun).

# displacement	--------    Dvigatel hajmi (odatda litr yoki kub santimetrda, masalan 2.0L).

# cylinders	---------    Dvigateldagi silindrlar soni (4, 6, 8 va hokazo).

# gears	---------    Uzatmalar qutisidagi tezliklar soni (masalan, 5 pog‘onali).

# transmission	-------   Uzatmalar qutisi turi (manual — qo‘lda, automatic — avtomat).

# mpg	--------    “Miles per gallon” — yonilg‘i tejamkorligi (1 gallon yonilg‘ida necha mil yuradi).

# aspiration	--------   Havo kirish turi: tabiiy (NA — naturally aspirated) yoki majburiy (turbo/supercharger).

# lockup_torque_converter	drive	--------  Avtomatik transmissiyada “lock-up” funksiyali moment o‘zgartirgich mavjudligi.

# max_ethanol   ---------    Yonilg‘idagi maksimal etanol foizi (masalan, E10 — 10% etanol).

# recommended_fuel	--------   Ishlab chiqaruvchi tavsiya qilgan yonilg‘i turi (Regular, Premium va hokazo).

# intake_valves_per_cyl	--------    Har bir silindrga to‘g‘ri keladigan kirish (havo/yonilg‘i) klapanlari soni.

# exhaust_valves_per_cyl	--------   Har bir silindrga to‘g‘ri keladigan chiqish (gaz) klapanlari soni.

# fuel_injection   ---------    Yonilg‘i purkash tizimi turi (port injection, direct injection va boshqalar).

# Kutubxonalarni chaqirish

In [10]:
import pandas as pd
import numpy as np

from joblib import dump
import os

import matplotlib.pyplot as plt
import seaborn as sns
import klib

from sklearn.model_selection import train_test_split, KFold,cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
labeller = LabelEncoder()

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

RF_Clas = RandomForestClassifier()
DT_Clas = DecisionTreeClassifier()
svc = SVC()

from sklearn.metrics import classification_report,accuracy_score

# Datani chaqirish

In [125]:
df = pd.read_csv(r"C:\Users\bunyo\Downloads\Telegram Desktop\cars2018.csv")

# Dataset bilan tanishamiz

In [75]:
df.sample(3)

,model,model_index,displacement,cylinders,gears,transmission,mpg,aspiration,lockup_torque_converter,drive,max_ethanol,recommended_fuel,intake_valves_per_cyl,exhaust_valves_per_cyl,fuel_injection
334,Mercedes-Benz CLA 250 4MATIC,212,2.0,4,7,Manual,27,Turbocharged/Supercharged,Y,4-Wheel Drive,10,Premium Unleaded Required,2,2,Direct ignition
16,"Ferrari North America, Inc. 488 Spider",144,3.9,8,7,Manual,18,Turbocharged/Supercharged,N,"2-Wheel Drive, Rear",10,Premium Unleaded Required,2,2,Direct ignition
1000,Mercedes-Benz GLA 250 4MATIC,505,2.0,4,7,Manual,26,Turbocharged/Supercharged,Y,4-Wheel Drive,10,Premium Unleaded Required,2,2,Direct ignition


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1144 entries, 0 to 1143
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   model                    1144 non-null   object 
 1   model_index              1144 non-null   int64  
 2   displacement             1144 non-null   float64
 3   cylinders                1144 non-null   int64  
 4   gears                    1144 non-null   int64  
 5   transmission             1144 non-null   object 
 6   mpg                      1144 non-null   int64  
 7   aspiration               1144 non-null   object 
 8   lockup_torque_converter  1144 non-null   object 
 9   drive                    1144 non-null   object 
 10  max_ethanol              1144 non-null   int64  
 11  recommended_fuel         1144 non-null   object 
 12  intake_valves_per_cyl    1144 non-null   int64  
 13  exhaust_valves_per_cyl   1144 non-null   int64  
 14  fuel_injection          

In [9]:
df.nunique()

model                      750
model_index                456
displacement                41
cylinders                    8
gears                        8
transmission                 3
mpg                         38
aspiration                   2
lockup_torque_converter      2
drive                        4
max_ethanol                  3
recommended_fuel             3
intake_valves_per_cyl        2
exhaust_valves_per_cyl       2
fuel_injection               2
dtype: int64

# Keraksiz ustunnni tashab yuboramiz masalan -> Model Index

In [126]:
df.drop(columns=['model_index'],inplace=True)

# Class yaratamiz

In [ ]:
class Preprocessing:
    def __init__(self,df):
        self.df=df
        
# Encoding qiluvchi
    def encoding_qilish(self):
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                if self.df[col].nunique() <= 2:
                    new_df = pd.get_dummies(self.df[col], prefix='New', dtype=int)
                    self.df.drop(columns=[col],inplace=True)
                    self.df = pd.concat([self.df,new_df],axis=1)
                else:
                    self.df[col] = labeller.fit_transform(self.df[col])
        return self
    
# Scale qiluvchi
    def scaling_qilish(self):
        for cols in self.df.columns:
            if self.df[cols].dtype != 'object' and self.df[cols].name != 'mpg':
                self.df[cols] = minmax.fit_transform(self.df[[cols]])
        return self
    
# To'ldruvchi
    def fillingNan(self):
        for cols in self.df.columns:
            if self.df[cols].isnull().any():
                if self.df[cols].dtype == 'object':
                    self.df[cols].fillna(self.df[cols].mode()[0], inplace=True)
                else:
                    self.df[cols].fillna(self.df[cols].mean(),inplace=True)
        return self          
    
# Skewness ni tog'irlash
    # def skw(self):
    #     for cols in self.df.columns:
    #         if self.df[cols].dtype != 'object':
    #             self.df[cols] = np.log()


# class orqali datani preprocess qilamiz

In [129]:
data_preprosesing = Preprocessing(df)
data_preprosesing.fillingNan().encoding_qilish().scaling_qilish()
preprocessed_data = data_preprosesing.df

In [130]:
preprocessed_data

,model,displacement,cylinders,gears,transmission,mpg,drive,max_ethanol,recommended_fuel,intake_valves_per_cyl,exhaust_valves_per_cyl,New_Naturally Aspirated,New_Turbocharged/Supercharged,New_N,New_Y,New_Direct ignition,New_Multipoint/sequential ignition
0,0.009346,0.357143,0.230769,0.888889,1.0,21,1.000000,0.000000,0.5,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
1,0.000000,0.114286,0.076923,0.555556,1.0,28,0.333333,0.000000,0.5,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
2,0.045394,0.600000,0.538462,0.666667,1.0,17,1.000000,0.066667,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0
3,0.046729,0.600000,0.538462,0.666667,1.0,18,0.333333,0.066667,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0
4,0.048064,0.600000,0.538462,0.666667,1.0,17,1.000000,0.066667,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1139,0.927904,0.671429,0.384615,0.777778,0.0,15,0.666667,0.066667,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0
1140,0.941255,0.671429,0.384615,0.555556,0.0,14,0.666667,0.066667,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0
1141,0.942590,0.671429,0.384615,0.555556,0.0,14,0.666667,1.000000,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0
1142,0.998665,0.142857,0.076923,0.777778,0.0,24,1.000000,0.000000,0.5,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0


In [136]:
preprocessed_data.skew()

model                            0.134643
displacement                     0.928599
cylinders                        1.227052
gears                           -1.180891
transmission                     0.878729
mpg                              1.450915
drive                            0.159967
max_ethanol                      4.119399
recommended_fuel                -0.496366
intake_valves_per_cyl           -3.250660
exhaust_valves_per_cyl          -3.156646
New_Naturally Aspirated          0.172187
New_Turbocharged/Supercharged   -0.172187
New_N                            1.382734
New_Y                           -1.382734
dtype: float64

# target qiymatga(mpg) juda kam korelatsya bo'gan ustunni tashlab yuboramiz -> Fuel Injection qiymatini

In [133]:
corr_df = preprocessed_data.corr()['mpg']
print(corr_df)

model                                 0.119912
displacement                         -0.740938
cylinders                            -0.713924
gears                                -0.385157
transmission                          0.244639
mpg                                   1.000000
drive                                -0.373795
max_ethanol                          -0.124257
recommended_fuel                      0.152029
intake_valves_per_cyl                 0.277959
exhaust_valves_per_cyl                0.291316
New_Naturally Aspirated               0.007729
New_Turbocharged/Supercharged        -0.007729
New_N                                 0.247939
New_Y                                -0.247939
New_Direct ignition                  -0.019907
New_Multipoint/sequential ignition    0.019907
Name: mpg, dtype: float64


In [134]:
preprocessed_data.drop(columns=['New_Direct ignition','New_Multipoint/sequential ignition'],inplace=True)

In [135]:
preprocessed_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1144 entries, 0 to 1143
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   model                          1144 non-null   float64
 1   displacement                   1144 non-null   float64
 2   cylinders                      1144 non-null   float64
 3   gears                          1144 non-null   float64
 4   transmission                   1144 non-null   float64
 5   mpg                            1144 non-null   int64  
 6   drive                          1144 non-null   float64
 7   max_ethanol                    1144 non-null   float64
 8   recommended_fuel               1144 non-null   float64
 9   intake_valves_per_cyl          1144 non-null   float64
 10  exhaust_valves_per_cyl         1144 non-null   float64
 11  New_Naturally Aspirated        1144 non-null   float64
 12  New_Turbocharged/Supercharged  1144 non-null   f